In [1]:
import torch

# 近似求导

In [7]:
def f(x):
    return 3. * x ** 2 + 2. * x + 1


def approximate_derivative(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2. * h)


print(approximate_derivative(f, 1.))

7.999999999785956


In [8]:
# 求偏导数
def g(x1, x2):
    return (x1 + 5) * (x2 ** 2)


def approximate_gradient(g, x1, x2, eps=1e-3):
    dg_x1 = approximate_derivative(lambda x: g(x, x2), x1, eps)
    # 使用匿名函数, 等价于将g(x, x2)赋值给f然后传入
    # 相当于固定了x2, 将x1作为自变量求导
    # 等价于
    # def f(x):
    #     return g(x, x2)
    dg_x2 = approximate_derivative(lambda x: g(x1, x), x2, eps)
    return dg_x1, dg_x2


print(approximate_gradient(g, 2., 3.))


(8.999999999993236, 41.999999999994486)


Torch自带的近似求导

In [9]:
# 声明两个tensor x1 和 x2，允许梯度计算，使用torch的自动求导上下文计算两个tensor的梯度
# 使用 torch.autograd.grad 计算 y = g(x1, x2) 的偏导数

x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
# 告诉 PyTorch: 这个变量需要参与求导(梯度计算)
# PyTorch 会跟踪所有基于它的运算
# 构建计算图
# 允许之后对它求导
y = g(x1, x2)

(dy_dx1,) = torch.autograd.grad(y, x1, retain_graph=True)
# 返回的是g对x1的偏导, retain_graph=True 表示计算后，保留计算图, 以便对同一个图继续求导
# torch.autograd.grad 返回的是一个tuple, 即使只求一个变量的导数也返回(grad_tensor,)
print(dy_dx1)

tensor([9.])


In [10]:
try:
    # 假如不加retain_graph=True, 第二次执行会报错，因为计算图已经被释放了
    (dy_dx2,) = torch.autograd.grad(y, x2, retain_graph=True)
    print(dy_dx2)
except Exception as e:
    print(e)

tensor([42.])


In [11]:
# 同时求导
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2])

print(dy_dx1, dy_dx2)

tensor([9.]) tensor([42.])


In [12]:
# 一般直接用backward
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

y.backward()
# backward()根据计算图计算偏导, 将结果自动放在x1.grad 和 x2.grad 中
print(x1.grad, x2.grad)

tensor([9.]) tensor([42.])


二阶导数

In [13]:
x1 = torch.tensor([2.], requires_grad=True)
x2 = torch.tensor([3.], requires_grad=True)
y = g(x1, x2)

# 求y对x1和x2的二阶偏导数
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2], create_graph=True)
dy_dx1_dx1, dy_dx1_dx2 = torch.autograd.grad(dy_dx1, [x1, x2], allow_unused=True)
# allow_unused = True意味着, 对没有参与计算图的导数求导时, 不报错而是返回None
dy_dx2_dx1, dy_dx2_dx2 = torch.autograd.grad(dy_dx2, [x1, x2], allow_unused=True)
print(dy_dx1_dx1, dy_dx2_dx1, dy_dx2_dx2)

None tensor([6.]) tensor([14.])


In [14]:
# 模拟梯度下降
learning_rate = 0.3
x = torch.tensor(2.0, requires_grad=True)
for _ in range(100):
    z = f(x)
    z.backward()
    x.data.sub_(learning_rate * x.grad)
    # x.data拿到的是x的值自身(还包含梯度等)
    # x -= learning_rate * x.grad
    # 这里就等价于optimizer.step()
    x.grad.zero_()  #梯度清零
print(x)

tensor(-0.3333, requires_grad=True)


In [16]:
a = torch.tensor(2)  # 标量
a.shape

torch.Size([])

In [15]:
learning_rate = 0.01
x = torch.tensor(2.0, requires_grad=True)
optimizer = torch.optim.SGD([x], lr=learning_rate, momentum=0.9)
for _ in range(500):
    z = f(x)
    optimizer.zero_grad()
    # 梯度变为0
    z.backward()
    # 求梯度
    optimizer.step()
    # x -= learning_rate * x.grad

print(x)

tensor(-0.3333, requires_grad=True)
